In [1]:
#Data Manipulation specific
import numpy as np
import pandas as pd
import xarray as xr
#import scipy.stats as stats
#from scipy import signal
from netCDF4 import Dataset
 
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap

import rasterio
from rasterio.transform import from_origin

import os

/home/anas.oubaha/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [5]:
!free -g

              total        used        free      shared  buff/cache   available
Mem:            376          53         148           1         173         317
Swap:             0           0           0


**Read NC data**

In [2]:
# Load dataset
dw = xr.open_dataset('data/vimdf_80N20S50W80E.nc', engine="netcdf4")  # Load NetCDF file
#dw = dw.sel(latitude=slice(45, 15))
#dw = dw.sel(valid_time=slice('1991-01-01', None))
#dw = dw[['r']] #keep only the variable r

In [3]:
dw

<xarray.Dataset>
Dimensions:     (valid_time: 408, latitude: 401, longitude: 521)
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 1991-01-01 1991-02-01 ... 2024-12-01
  * latitude    (latitude) float64 80.0 79.75 79.5 79.25 ... -19.5 -19.75 -20.0
  * longitude   (longitude) float64 -50.0 -49.75 -49.5 ... 79.5 79.75 80.0
    expver      (valid_time) object ...
Data variables:
    vimdf       (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

**Seasonal Relative Humidity anomalies**

In [3]:
# Convert 'date' from integer to datetime format
dw = dw.assign_coords(date=pd.to_datetime(dw['valid_time'].data.astype(str), format='ISO8601'))

# Convert dataset to DataFrame
df = dw['vimdf'].to_dataframe().reset_index()

# Extract year and month
df = df.drop('expver', axis=1)
df = df.drop('number', axis=1)
df['year'] = df['valid_time'].dt.year
df['month'] = df['valid_time'].dt.month

# **FAST NumPy Vectorized Method for Assigning Seasons**
seasons = np.array(["DJF", "MAM", "JJA", "SON"])

# Define conditions for season assignment
winter = df['month'].isin([12, 1, 2])
spring = df['month'].isin([3, 4, 5])
summer = df['month'].isin([6, 7, 8])
autumn = df['month'].isin([9, 10, 11])

# Assign seasons using np.select (much faster than apply)
df['season'] = np.select([winter, spring, summer, autumn], seasons)

# Assign season_year (December belongs to next year’s winter)
df['season_year'] = df['year']
df.loc[df['month'] == 12, 'season_year'] += 1  # Shift December to next year's winter

# **STEP 1: Compute Actual Seasonal SST (For Each Year)**
seasonal_vimdf_df = df.groupby(['latitude', 'longitude', 'season_year', 'season'])['vimdf'].mean().reset_index()

# **STEP 2: Compute Seasonal Climatology (Long-Term Average Over All Years)**
# Filter to only include seasons from 1991 to 2020 (30 years period recommended by the WMO)
filtered_df = seasonal_vimdf_df[(seasonal_vimdf_df['season_year'] >= 1991) & (seasonal_vimdf_df['season_year'] <= 2020)]

seasonal_climatology_df = filtered_df.groupby(['latitude', 'longitude', 'season'])['vimdf'].mean().reset_index()
seasonal_climatology_df.rename(columns={'vimdf': 'vimdf_climatology'}, inplace=True)

# **STEP 3: Merge Actual SST With Climatology**
mergedf = pd.merge(seasonal_vimdf_df, seasonal_climatology_df, on=['latitude', 'longitude', 'season'], how="left")
# **STEP 4: Compute Seasonal Anomalies**
mergedf['vimdf_anomaly'] = mergedf['vimdf'] - mergedf['vimdf_climatology']

mergedf['date'] = (
    mergedf['season'].astype(str) 
    + ' ' 
    + mergedf['season_year'].astype(str)
)
#########################################################################################
from scipy.stats import ttest_ind

# Define drought years
drought_years = [1994, 1995, 1999, 2000, 2001, 2005, 2016, 2019, 2020, 2021, 2022, 2023]  # Replace with your own

# Create a unique season key
#mergedf['season_key'] = mergedf['season'] + '_' + mergedf['season_year'].astype(str)

# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedf.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='vimdf_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
#t_df = pd.DataFrame([t_vals], columns=pivot.columns).T.reset_index()
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
#t_df.columns = ['latitude', 'longitude', 'season', 't_value']
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge t/p values back
#significance_df = pd.merge(t_df, p_df, on=['latitude', 'longitude', 'season'])

# Merge with original mergedf
mergedf = pd.merge(mergedf, p_df, on=['latitude', 'longitude', 'season'], how='left')
##########################################################################################
# Convert back to xarray
anomalies = mergedf.set_index(['date', 'latitude', 'longitude'])['vimdf_anomaly'].to_xarray()
pval = mergedf.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()
anomalies

<xarray.DataArray 'vimdf_anomaly' (date: 137, latitude: 401, longitude: 521)>
array([[[-3.71602364e-05, -4.04916391e-05, -4.48086721e-05, ...,
          6.31969488e-06,  1.59006595e-06, -3.71276110e-06],
        [-3.46478992e-05, -3.80806305e-05, -4.70087471e-05, ...,
          8.57095074e-07, -4.79508162e-06, -8.58345447e-06],
        [-4.09991080e-05, -4.45401238e-05, -4.84003504e-05, ...,
         -2.75080856e-06, -1.00861525e-05, -1.72572545e-05],
        ...,
        [-4.61657919e-07, -4.90301318e-07, -5.44773229e-07, ...,
          8.77128969e-07,  1.06985044e-06,  1.23575001e-06],
        [ 1.66674340e-07,  3.28269266e-07,  4.99466978e-07, ...,
          4.20821834e-07,  5.54104645e-07,  6.38544407e-07],
        [ 4.43193358e-08,  2.04755111e-07,  3.79926632e-07, ...,
         -1.50057986e-07, -1.74377419e-08,  1.27600288e-07]],

       [[-1.25562074e-05, -4.12065856e-06, -1.09638859e-06, ...,
         -3.24558823e-05, -3.21787193e-05, -3.25542314e-05],
        [-1.94117310e-05,  1.19110700e-06,  7.61118645e-06, ...,
         -3.43869069e-05, -3.60455706e-05, -3.67295361e-05],
        [-1.68719162e-05, -5.29813406e-07,  5.24107600e-06, ...,
         -3.92018046e-05, -4.15104878e-05, -4.41913762e-05],
...
          2.79055530e-06,  2.96870712e-06,  3.14652743e-06],
        [-1.17845138e-06, -1.29534283e-06, -1.36918652e-06, ...,
          4.24822065e-06,  4.05715446e-06,  3.87933324e-06],
        [-7.92677156e-07, -8.54930931e-07, -9.15197916e-07, ...,
          3.96178621e-06,  3.79389985e-06,  3.59521755e-06]],

       [[-2.70565615e-05, -3.37153942e-05, -2.54995593e-05, ...,
         -4.59322837e-06, -6.22208972e-06, -5.59491673e-06],
        [-2.36554552e-05, -2.93225312e-05, -2.36425421e-05, ...,
         -8.38275810e-06, -8.44699753e-06, -7.84963049e-06],
        [-1.98497000e-05, -3.08547023e-05, -1.77406891e-05, ...,
         -9.42087354e-06, -9.25431232e-06, -7.77147943e-06],
        ...,
        [-7.51683638e-06, -7.07940490e-06, -6.63965511e-06, ...,
         -9.63891580e-08, -2.16261014e-07, -3.06661605e-07],
        [-6.42971418e-06, -6.48700006e-06, -6.53104189e-06, ...,
          6.75822321e-07,  7.03306796e-07,  7.44035788e-07],
        [-5.74591559e-06, -5.78830213e-06, -5.82870052e-06, ...,
         -7.97626853e-09,  1.28857209e-08,  3.27536327e-08]]],
      dtype=float32)
Coordinates:
  * date       (date) object 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 -20.0 -19.75 -19.5 -19.25 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 -50.0 -49.75 -49.5 -49.25 ... 79.5 79.75 80.0

In [4]:
import os
def export_to_geotiff(path, u_data, v_data, lat, lon, description):
    # Flip latitude if needed (GeoTIFF expects top-to-bottom order)
    if lat[0] > lat[-1]:
        u_data = u_data[::-1]
        v_data = v_data[::-1]
        lat = lat[::-1]

    # Get transform
    pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1)
    pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1)
    transform = from_origin(lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y)

    # Stack u and v into a 3D array (bands, height, width)
    data = np.stack([u_data, v_data])  # shape: (2, lat, lon)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=u_data.shape[0],
        width=u_data.shape[1],
        count=2,  # 2 bands: u and v
        dtype=u_data.dtype,
        crs="EPSG:4326",
        transform=transform,
        description=description
    ) as dst:
        dst.write(data[0], 1)  # Band 1 - u
        dst.write(data[1], 2)  # Band 2 - v
        
# Export regression slope map
output_dir = 'VIMDF'  # no leading slash unless it's an absolute path
os.makedirs(output_dir, exist_ok=True)

for d in range(137):
    file_path = os.path.join(output_dir, str(anomalies['date'][d].values) + '.tif')
    export_to_geotiff(file_path, anomalies[d], pval[d], anomalies.latitude.values, anomalies.longitude.values, 'VIMDF Seasonal Anomalies of '+str(anomalies['date'][d].values))